In [ ]:
import sys, importlib, os

_src_training = os.path.dirname(os.path.abspath("training.ipynb"))
_src = os.path.dirname(_src_training)
for _p in [_src_training, _src]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import infrastructure, pipeline, classifier, evaluation
for _mod in [infrastructure, pipeline, classifier, evaluation]:
    importlib.reload(_mod)

from infrastructure import clean_neo4j_db, clean_kafka_topics, delete_test_neo4j_nodes, verify_concept_created
from pipeline import train_mnist, remove_concept, retrain_concept
from evaluation import test_mnist_all
from classifier import classify_image
import json, uuid
print("Modules loaded.")

In [ ]:
classes_to_subclasses = {
    0: [1],
    1: [1, 3],
    2: [1, 2],
    3: [1],
    4: [1, 2],
    5: [1],
    6: [1],
    7: [1],
    8: [1],
    9: [2],
}

In [ ]:
clean_neo4j_db()
# clean_kafka_topics()

for class_num in classes_to_subclasses:
    for subclass in classes_to_subclasses[class_num]:
        train_mnist(class_number=class_num, subclass=subclass, is_prepared_samples=True, with_concept_creation=True)

In [ ]:
params = {
    "ged_timeout": 5,
    "skeletonization_threshold": 180,
    "comparison_method": "fgw",
    "fgw_alpha": 0.6,
    "property_normalizers": {
        "normalized_x": 3.0,
        "normalized_y": 3.0,
        "horizontal_direction": 2.0,
        "vertical_direction": 2.0,
        "cycle_count": 1.0,
        "angle_with_ox": 45,
    },
}
results, y_true, y_pred, run_dir = test_mnist_all(
    classes=list(classes_to_subclasses.keys()),
    params=params,
    sample_fraction=1.0,
    description="Testing after retraining with the new concept formation with fgw method",
)

In [ ]:
import optuna
import mlflow
from sklearn.metrics import accuracy_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Configuration ──────────────────────────────────────────
N_TRIALS = 60
N_STARTUP_TRIALS = 20
SAMPLE_FRACTION = 0.25  # use 25% of data for fast proxy evaluation
STUDY_NAME = "naturalagi-hp-tuning"
TUNE_COMPARISON_METHOD = True  # set True to also search over ged vs fgw

OPTUNA_MLFLOW_EXPERIMENT = "naturalagi-hp-tuning"

def objective(trial: optuna.Trial) -> float:
    comparison_method = "ged"
    fgw_alpha = 0.5
    ged_timeout = 5

    if TUNE_COMPARISON_METHOD:
        comparison_method = trial.suggest_categorical("comparison_method", ["ged", "fgw"])
        if comparison_method == "fgw":
            fgw_alpha = trial.suggest_float("fgw_alpha", 0.01, 0.99)
        else:
            ged_timeout = trial.suggest_float("ged_timeout", 1, 15)
    else:
        ged_timeout = trial.suggest_float("ged_timeout", 1, 15)

    # Delta parameterization: guarantees MINOR < GENERAL < SEVERE
    # while keeping each parameter's semantic role stable for TPE
    cost_minor = trial.suggest_float("cost_minor", 0.05, 0.4)
    gap_mg = trial.suggest_float("gap_minor_general", 0.05, 0.3)
    gap_gs = trial.suggest_float("gap_general_severe", 0.05, 0.3)
    cost_general = cost_minor + gap_mg
    cost_severe = cost_general + gap_gs

    cost_no_match = trial.suggest_float("cost_no_match", 0.8, 2.0)
    cost_impossible = trial.suggest_float("cost_impossible", 10.0, 200.0, log=True)

    params = {
        "comparison_method": comparison_method,
        "ged_timeout": ged_timeout,
        "fgw_alpha": fgw_alpha,
        "skeletonization_threshold": trial.suggest_int("skel_threshold", 100, 220, step=10),
        "simplification_epsilon": trial.suggest_float("simplification_epsilon", 0.5, 5.0),
        "property_normalizers": {
            "normalized_x": trial.suggest_float("norm_x", 0.5, 5.0),
            "normalized_y": trial.suggest_float("norm_y", 0.5, 5.0),
            "horizontal_direction": trial.suggest_float("norm_hdir", 0.5, 4.0),
            "vertical_direction": trial.suggest_float("norm_vdir", 0.5, 4.0),
            "cycle_count": trial.suggest_float("norm_cycle", 0.5, 8.0),
            "angle_with_ox": trial.suggest_float("norm_angle", 10.0, 360.0),
        },
        "node_costs": {
            "NO_COST": 0.0,
            "MINOR": cost_minor,
            "GENERAL": cost_general,
            "SEVERE": cost_severe,
            "NO_MATCH": cost_no_match,
            "IMPOSSIBLE": cost_impossible,
        },
    }

    _, y_true, y_pred, _ = test_mnist_all(
        classes=list(classes_to_subclasses.keys()),
        params=params,
        sample_fraction=SAMPLE_FRACTION,
        description=f"optuna trial {trial.number}",
    )
    return accuracy_score(y_true, y_pred)


# ── Redirect MLflow to separate experiment ─────────────────
_original_experiment = evaluation.MLFLOW_EXPERIMENT
evaluation.MLFLOW_EXPERIMENT = OPTUNA_MLFLOW_EXPERIMENT

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5050"))
mlflow.set_experiment(OPTUNA_MLFLOW_EXPERIMENT)

try:
    with mlflow.start_run(run_name=f"optuna_{STUDY_NAME}_{N_TRIALS}trials") as parent_run:
        mlflow.log_params({
            "n_trials": N_TRIALS,
            "n_startup_trials": N_STARTUP_TRIALS,
            "sample_fraction": SAMPLE_FRACTION,
            "tune_comparison_method": TUNE_COMPARISON_METHOD,
            "sampler": "TPESampler",
        })

        # ── Create study & optimize ────────────────────────
        study = optuna.create_study(
            study_name=STUDY_NAME,
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=N_STARTUP_TRIALS),
        )
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

        # ── Log best results to parent run ─────────────────
        mlflow.log_metric("best_accuracy", study.best_value)
        mlflow.log_params({f"best.{k}": v for k, v in study.best_params.items()})

        try:
            importances = optuna.importance.get_param_importances(study)
            for param, imp in importances.items():
                mlflow.log_metric(f"importance.{param}", imp)
        except Exception:
            importances = {}

    # ── Print results ──────────────────────────────────────
    print(f"\n{'='*50}")
    print(f"Best accuracy: {study.best_value*100:.2f}%")
    print(f"Best params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
    print(f"{'='*50}")

    if importances:
        print("\nParameter importance:")
        for param, imp in importances.items():
            bar = "█" * int(imp * 30)
            print(f"  {param:20s} {imp:.3f} {bar}")

finally:
    evaluation.MLFLOW_EXPERIMENT = _original_experiment

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

delete_test_neo4j_nodes()

class_number = 1
img_num = 448
image_id = f"mnist_{class_number}_{img_num:05d}"
local_path = f"../../tests/generated_samples/mnist_{class_number}/test"
nuclio_path = f"/opt/nuclio/shared_storage/generated_samples/mnist_{class_number}/test"

img_file = f"{image_id}.png"
img_local = os.path.join(local_path, img_file)
if os.path.exists(img_local):
    img = mpimg.imread(img_local)
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    plt.title(f"MNIST Class {class_number}, Image #{img_num}")
    plt.axis("off")
    plt.show()

image_id_for_test = str(uuid.uuid4())
params = {
    "ged_timeout": 5,
    "skeletonization_threshold": 180,
    "image_id": image_id_for_test,
    "session_id": "test",
    "delete_image_nodes": False,
    # "simplification_epsilon": 5,
}
result = classify_image(os.path.join(nuclio_path, img_file), params=params, timeout=60)
print(json.dumps(result, indent=2))

In [ ]:
# remove_concept("3_1")
retrain_concept(number=2, subclass=2, with_concept_creation=True)